# Exploring the Dataset: VPN (OpenVPN) Log

**Goal:** Understand the structure of `vpn/logs/openvpn.log` (OpenVPN server log) to design the `vpn_events` table.

This notebook walks through:
1. Loading the raw OpenVPN log (5,537 lines of `YYYY-MM-DD HH:MM:SS client message` format)
2. Parsing the format and exploring every field (including parsed client and message_prefix for analysis)
3. Building a raw 1:1 DataFrame (`df_raw`) for DDL; keeping parsed columns in `df` for field-by-field exploration only
4. Integrating ground truth labels (28 labeled lines, Initial Access / attacker VPN)
5. Mapping to the planned raw table schema with both PostgreSQL and MySQL DDL
6. Checking for 1NF, 2NF, and 3NF violations using the `normalization_rules_sheet.md` checklist

---

**Dataset:** AIT Log Data Set V2.0 — russellmitchell testbed
**Source:** https://zenodo.org/records/5789064
**Host:** vpn (VPN gateway, DMZ)


## 0. Configuration

Set the path to the `russellmitchell/` dataset folder below.

**Default:** Assumes `russellmitchell/` is at the same level as the repo. We also need the label file path for ground-truth attack lines.


In [ ]:
from pathlib import Path

# --- CHANGE THIS if your dataset is in a different location ---
DATASET_ROOT = Path("..") / ".." / "russellmitchell"
VPN_LOG = DATASET_ROOT / "gather" / "vpn" / "logs" / "openvpn.log"
LABEL_FILE = DATASET_ROOT / "labels" / "vpn" / "logs" / "openvpn.log"

for name, path in [
    ("Dataset root", DATASET_ROOT),
    ("VPN log", VPN_LOG),
    ("Label file", LABEL_FILE),
]:
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{name}: {path.resolve()} [{status}]")


## 1. Load Raw Data

The OpenVPN log uses one line per event:
```
YYYY-MM-DD HH:MM:SS client message
```
Where `client` is either `username/IP:port` (after auth) or `IP:port` (initial connection). Example: `2022-01-24 03:01:00 192.168.230.122:53581 TLS: Initial packet from [AF_INET]192.168.230.122:53581, sid=...`

Load all lines with 1-based line numbers (to match the label file).


In [ ]:
# Load all lines with 1-based line numbers (matching label file convention)
with open(VPN_LOG) as f:
    raw_lines = f.readlines()

print(f"Total lines: {len(raw_lines)}")
print()
print("First 5 lines:")
for i, line in enumerate(raw_lines[:5], 1):
    print(f"  [{i}] {line.rstrip()}")
print()
print("Last 3 lines:")
for i, line in enumerate(raw_lines[-3:], len(raw_lines) - 2):
    print(f"  [{i}] {line.rstrip()}")


## 2. Parse the OpenVPN Log Format

### 2.1 Message prefix (event type) inventory

First, extract the leading token or phrase of each message and count. These drive event semantics (VERIFY OK, peer info, TLS, Control Channel, MULTI, etc.).


In [ ]:
import re
from collections import Counter

# Message is everything after "YYYY-MM-DD HH:MM:SS client "; take first token(s) as prefix
prefix_counts = Counter()
for line in raw_lines:
    line = line.rstrip()
    m = re.match(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2} \S+ (.+)", line)
    if m:
        msg = m.group(1)
        # First token (or first part up to ": " for "VERIFY OK:")
        if ": " in msg and not msg.startswith("peer info"):
            prefix = msg.split(": ")[0].strip()
        else:
            prefix = msg.split()[0] if msg else ""
        if prefix:
            prefix_counts[prefix] += 1

print(f"Distinct message prefixes: {len(prefix_counts)}")
print()
for prefix, count in prefix_counts.most_common(20):
    print(f"  {prefix:45s} {count:5d} ({count / len(raw_lines) * 100:5.1f}%)")


### 2.2 Sample line per message prefix

Examine one example of each major prefix to understand message structure.


In [ ]:
# One example per prefix (top 15)
prefix_examples = {}
for line in raw_lines:
    line = line.rstrip()
    m = re.match(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2} \S+ (.+)", line)
    if m:
        msg = m.group(1)
        if ": " in msg and not msg.startswith("peer info"):
            prefix = msg.split(": ")[0].strip()
        else:
            prefix = msg.split()[0] if msg else ""
        if prefix and prefix not in prefix_examples:
            prefix_examples[prefix] = line

for prefix in [p for p, _ in prefix_counts.most_common(15)]:
    if prefix in prefix_examples:
        print(f"--- {prefix} ---")
        print(f"  {prefix_examples[prefix][:100]}...")
        print()


### 2.3 Parsing strategy

| Component | Pattern | Raw column | Parsed (analysis only) |
|-----------|---------|------------|------------------------|
| Timestamp | YYYY-MM-DD HH:MM:SS | event_timestamp | — |
| Client | username/IP:port or IP:port | client | username, client_ip, client_port |
| Message | rest of line | message | message_prefix (first token/phrase) |

The **raw** table stores `client` and `message` as-is. Parsed columns (username, client_ip, client_port, message_prefix) live in `df` for exploration; DDL matches `df_raw`.


In [ ]:
import re

import pandas as pd

# Match: YYYY-MM-DD HH:MM:SS client message
LINE_RX = re.compile(r"^(\d{4}-\d{2}-\d{2}) (\d{2}:\d{2}:\d{2}) (\S+) (.*)$")


def parse_vpn_line(line_num, line):
    """Parse one openvpn.log line. Returns dict with line_number, event_timestamp,
    client, message (raw), and parsed fields for analysis only: username, client_ip, client_port, message_prefix.
    """
    record = {"line_number": line_num}
    line = line.rstrip()
    m = LINE_RX.match(line)
    if not m:
        return record
    date_str, time_str, client, message = m.groups()
    record["client"] = client
    record["message"] = message.strip() if message else ""

    # Timestamp
    ts_str = f"{date_str} {time_str}"
    try:
        record["event_timestamp"] = pd.to_datetime(ts_str)
    except Exception:
        record["event_timestamp"] = None

    # --- Parsed sub-fields (analysis only; not in raw DDL) ---
    # username from client when format is user/IP:port
    if "/" in client:
        record["username"] = client.split("/")[0].strip()
        rest = client.split("/", 1)[1]
    else:
        record["username"] = None
        rest = client
    # client_ip and client_port from rest (IP:port or already just IP)
    if ":" in rest:
        record["client_ip"] = rest.rsplit(":", 1)[0].strip()
        try:
            record["client_port"] = int(rest.rsplit(":", 1)[1])
        except ValueError:
            record["client_port"] = None
    else:
        record["client_ip"] = rest.strip() or None
        record["client_port"] = None

    # message_prefix: first token or part before ": " for things like "VERIFY OK:"
    if record["message"]:
        msg = record["message"]
        if ": " in msg and not msg.startswith("peer info"):
            record["message_prefix"] = msg.split(": ")[0].strip()
        else:
            record["message_prefix"] = msg.split()[0] if msg.split() else ""
    else:
        record["message_prefix"] = None

    return record


parsed = [parse_vpn_line(i, line) for i, line in enumerate(raw_lines, 1)]
print(f"Parsed {len(parsed)} records")
print(f"Sample (line 1): {parsed[0]}")


In [ ]:
df = pd.DataFrame(parsed)

if "client_port" in df.columns:
    df["client_port"] = df["client_port"].astype("Int64")

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
parsed_cols = [c for c in df.columns if c in ("username", "client_ip", "client_port", "message_prefix")]
print(f"Parsed columns (analysis only): {parsed_cols}")


## 3. Field-by-Field Exploration

### 3.1 client (and parsed username / client_ip)


In [ ]:
print("=== client value counts (top 15) ===")
print(df["client"].value_counts().head(15).to_string())
print()
if "username" in df.columns:
    print("=== username (non-null count) ===")
    print(f"  {df['username'].notna().sum()} / {len(df)}")
if "client_ip" in df.columns:
    print("=== client_ip (top 10) ===")
    print(df["client_ip"].value_counts().head(10).to_string())


### 3.2 event_timestamp


In [ ]:
print("=== event_timestamp range ===")
print(f"  Earliest: {df['event_timestamp'].min()}")
print(f"  Latest:   {df['event_timestamp'].max()}")
print()
print("=== Events per day ===")
dates = df["event_timestamp"].dt.date
for date, count in dates.value_counts().sort_index().items():
    print(f"  {date}  {count:5d}")


### 3.3 message_prefix distribution


In [ ]:
print("=== message_prefix (top 15) ===")
print(df["message_prefix"].value_counts().head(15).to_string())


### 3.4 High-signal: attacker IP 192.168.230.122

From findings: attacker external IP is 192.168.230.122; 28 labeled lines (4331–4358) are this connection.


In [ ]:
attacker_ip = "192.168.230.122"
attacker_mask = df["client"].str.contains(attacker_ip, na=False) | (
    df["client_ip"] == attacker_ip if "client_ip" in df.columns else False
)
df_attacker = df[attacker_mask]
print(f"Lines where client/client_ip is {attacker_ip}: {len(df_attacker)}")
print()
print(df_attacker[["line_number", "event_timestamp", "client", "message_prefix"]].head(10).to_string())


## 4. Raw 1:1 DataFrame

Build `df_raw` by dropping parsed columns (username, client_ip, client_port, message_prefix). The raw table DDL must match `df_raw` (client and message as single columns).


In [ ]:
parsed_cols = [c for c in df.columns if c in ("username", "client_ip", "client_port", "message_prefix")]
df_raw = df.drop(columns=[c for c in parsed_cols if c in df.columns])

print(f"df shape: {df.shape}")
print(f"df_raw shape: {df_raw.shape}")
print(f"Raw columns: {list(df_raw.columns)}")
print()
print("Null counts in df_raw:")
for col in df_raw.columns:
    n = df_raw[col].isnull().sum()
    if n > 0:
        print(f"  {col}: {n}")


In [ ]:
print("=== First 5 rows (df_raw) ===")
print(df_raw.head().to_string())
print()
print("=== Last 3 rows (df_raw) ===")
print(df_raw.tail(3).to_string())


## 5. Ground Truth Labels

The label file maps line numbers to attack labels (JSONL). 28 labeled lines (4331–4358): Initial Access — attacker VPN connection from 192.168.230.122 (authenticated as jhall).


In [ ]:
import json

labels = []
if LABEL_FILE.exists():
    with open(LABEL_FILE) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            labels.append(json.loads(line))

print(f"Labeled records: {len(labels)}")
if labels:
    print(f"Labeled line numbers: {[lbl['line'] for lbl in labels][:5]} ... {[lbl['line'] for lbl in labels][-2:]}")
    print()
    for lbl in labels[:3]:
        print(f"  line {lbl['line']}: labels={lbl['labels']}, rules={list(lbl['rules'].keys())}")
    if len(labels) > 3:
        print("  ...")
else:
    print("Label file not found or empty.")


In [ ]:
# Cross-reference labeled lines with df_raw
if labels:
    labeled_line_nums = {lbl["line"] for lbl in labels}
    df_labeled = df_raw[df_raw["line_number"].isin(labeled_line_nums)]
    print(f"Labeled records matched in df_raw: {len(df_labeled)}")
    print()
    print(df_labeled[["line_number", "event_timestamp", "client", "message"]].head(5).to_string())
else:
    df_labeled = df_raw.iloc[0:0]
    print("No labels loaded.")


In [ ]:
# Label distribution (findings: attacker_vpn 28, foothold 28)
if labels:
    from collections import Counter
    label_counter = Counter()
    for lbl in labels:
        for name in lbl["labels"]:
            label_counter[name] += 1
    print("=== Label distribution ===")
    for name, count in label_counter.most_common():
        print(f"  {name}: {count}")
    print()
    print("=== Raw log lines for first 2 labeled events ===")
    for lbl in labels[:2]:
        line_num = lbl["line"]
        print(f"  [{line_num}] {raw_lines[line_num - 1].rstrip()}")
        print(f"         labels: {lbl['labels']}")
        print()


## 6. Schema Mapping

Map the parsed fields to the planned **raw** table `vpn_events_raw`. The table stores `client` and `message` as single columns (1NF violation when message embeds multiple attributes). Parsed columns are for analysis only and are **not** in the raw DDL.

### 6.1 Raw field → column mapping

| # | Raw field | DB Column | PostgreSQL Type | MySQL Type | Nullable | Notes |
|---|-----------|-----------|-----------------|------------|----------|-------|
| 1 | *(auto)* | vpn_event_id | SERIAL PRIMARY KEY | INT AUTO_INCREMENT PRIMARY KEY | No | Surrogate key |
| 2 | (line index) | line_number | INTEGER NOT NULL | INT NOT NULL | No | 1-based; candidate key; joins to labels |
| 3 | timestamp | event_timestamp | TIMESTAMP WITH TIME ZONE | DATETIME | Yes | Full date+time from line |
| 4 | client | client | VARCHAR(100) NOT NULL | VARCHAR(100) NOT NULL | No | username/IP:port or IP:port |
| 5 | message | message | TEXT NOT NULL | TEXT NOT NULL | No | Raw blob (1NF violation when composite) |

Optional columns when loading with labels: `vpn_event_category` (TEXT[]/JSON), `vpn_signature_matches` (JSONB/JSON). See hunt_vpn_logs_findings.md.

### 6.2 Raw DDL


In [ ]:
postgresql_ddl = '''
-- PostgreSQL
CREATE TABLE vpn_events_raw (
    vpn_event_id          SERIAL PRIMARY KEY,
    line_number           INTEGER NOT NULL,
    event_timestamp       TIMESTAMP WITH TIME ZONE,
    client                VARCHAR(100) NOT NULL,
    message               TEXT NOT NULL,
    vpn_event_category    TEXT[],
    vpn_signature_matches JSONB,
    created_at            TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
'''

mysql_ddl = '''
-- MySQL
CREATE TABLE vpn_events_raw (
    vpn_event_id          INT AUTO_INCREMENT PRIMARY KEY,
    line_number           INT NOT NULL,
    event_timestamp       DATETIME,
    client                VARCHAR(100) NOT NULL,
    message               TEXT NOT NULL,
    vpn_event_category    JSON,
    vpn_signature_matches JSON,
    created_at            DATETIME DEFAULT CURRENT_TIMESTAMP
);
'''

print(postgresql_ddl)
print(mysql_ddl)


## 7. Observations for Normalization Phase

Use the `normalization_rules_sheet.md` checklist.

### 7.1 1NF Check

- **Multi-valued / composite field (message):** The `message` column often contains multiple attributes in one cell (e.g. `VERIFY OK: depth=1, C=AT, ST=Vienna, ... CN=OpenVPN CA`; `TLS: soft reset sec=3308/3308 bytes=45748/-1 pkts=649/0`). **1NF violated.** Resolution: retain raw message; optionally parse into atomic columns during normalization.
- **Multi-valued fields (labels file):** `labels` (array) and `rules` (nested dict) are 1NF violations; store as-is in raw load, unpack to `attack_labels` in normalization.
- **Repeating groups:** None.

### 7.2 2NF Check

- The table has a **single-column primary key** (`vpn_event_id`). **2NF is satisfied.**

### 7.3 3NF Check

- **Transitive dependency:** The message prefix (event type) determines which sub-fields can be parsed from the rest of the message (VERIFY OK → depth, CN, ...; peer info → IV_VER, ...). So vpn_event_id → message_type → populated field set. **3NF violated.** Resolution: subtype tables or type-specific columns in final 3NF schema.

### 7.4 Preliminary Functional Dependencies

| FD | Determinant | Dependent(s) | Reasoning |
|----|-------------|---------------|-----------|
| FD1 | vpn_event_id | all other attributes | Surrogate PK. |
| FD2 | line_number | all other attributes | Each log line is unique; candidate key. |
| FD3 | message prefix / event type | populated field set | Message type determines which attributes can be parsed (3NF violation). |
| FD4 | client | constant per connection | Within a session, client is fixed; meaningful for grouping. |


## 8. Summary

### What we discovered

1. openvpn.log has 5,537 lines; format `YYYY-MM-DD HH:MM:SS client message` (client = username/IP:port or IP:port).
2. **1NF violated:** `message` is often composite (key=value lists); labels file has array and nested dict.
3. **2NF satisfied:** single-column PK.
4. **3NF violated:** message prefix (event type) determines which sub-fields exist (type → field set).
5. 28 labeled lines (4331–4358) are the attacker VPN connection from 192.168.230.122 (Initial Access); labels `attacker_vpn`, `foothold`.

### Raw schema output

- `df_raw`: 4 columns (line_number, event_timestamp, client, message) — DDL matches this; table name `vpn_events_raw`. Optional label columns when loading with labels.
- Parsed columns (username, client_ip, client_port, message_prefix) live in `df` for analysis only.

### Next steps

- Consolidate findings in `docs/data_exploration/notebook_findings/hunt_vpn_logs_findings.md`.
- In normalization: optionally split client into username/client_ip/client_port; optionally parse message into atomic columns; resolve type → field set; cross-log correlation with 172.19.131.174 (VPN-assigned attacker IP).
